# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Not subscripting or iterating, just reference
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = dataset.record_sets

print("Record sets available (referenced by `@id`):")
for rec in record_sets:
    print(f"- @id: {rec['@id']}, name: {rec.get('name', 'N/A')}")

# For each record set, print its fields
for rec in record_sets:
    print(f"\nRecord set: {rec.get('name', 'N/A')} (@id: {rec['@id']})")
    fields = rec.get('fields', []) if 'fields' in rec else []
    if not fields:
        print('  [No fields listed]')
        continue
    for field in fields:
        field_id = field.get('@id') if isinstance(field, dict) else field
        print(f"  Field @id: {field_id}")

# Show a few records from each record set
print("\nSample records from the first record set:")
if record_sets:
    sample_recset_id = record_sets[0]['@id']
    for i, rec in enumerate(dataset.records(record_set=sample_recset_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather all record set @ids for extraction
record_set_ids = [rec['@id'] for rec in record_sets]
dataframes = {}

for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {df.shape[0]} records from record set @id: {rec_id}")
    except Exception as e:
        print(f"Could not load from record set @id: {rec_id}: {e}")

if dataframes:
    # To demonstrate, use the first record set for further analysis
    active_recset_id = record_set_ids[0]
    print(f"Columns in DataFrame for record set @id {active_recset_id}:")
    print(dataframes[active_recset_id].columns.tolist())
    display(dataframes[active_recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Inspect numeric columns for EDA
numeric_candidates = []
df = dataframes[active_recset_id]
for col in df.columns:
    try:
        # Check if most values could be numbers (heuristic)
        sample = pd.to_numeric(df[col].dropna().head(20), errors='coerce')
        if sample.notna().mean() > 0.7:
            numeric_candidates.append(col)
    except Exception:
        continue

print(f"Numeric candidate fields (@id): {numeric_candidates}")

# Pick the first available numeric field for demonstration
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # This should be a column @id
else:
    print("No numeric fields found.")
    numeric_field = None

if numeric_field and numeric_field in df.columns:
    # Remove NaNs and filter by threshold (arbitrary, e.g., top 10% quantile)
    threshold = df[numeric_field].quantile(0.9)
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field]].head())
    
    # Normalize
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (if available)
    cat_candidates = [col for col in df.columns if df[col].nunique() < 15 and col != numeric_field]
    if cat_candidates:
        group_field = cat_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No categorical grouping field found.")
else:
    print("No valid numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {active_recset_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No available numeric field to visualize.")

## 6. Conclusion
This notebook demonstrated how to:
- Load Croissant dataset metadata and records with `mlcroissant`.
- Enumerate and reference record sets and fields by their `@id`.
- Extract data to pandas DataFrames for analysis.
- Conduct simple EDA steps: filtering, normalization, grouping, and visualization.

You can extend this analysis by examining additional record sets, further exploring non-numeric features, or integrating with external analysis workflows using the power of open scientific data!